# Public Transit Across the US

Day 18 · NULL. US counties with no stop on any fixed-route service: no scheduled local or regional bus or rail stop,
no Amtrak stop, no intercity bus or rail terminal, no ferry terminal. Counties are then split by whether any agency
reports dial-a-ride (demand response) there to the National Transit Database, with population for every county.

Revised Nov 20, 2026 after comments on the first version ("Counties Without Public Transit"): dial-a-ride and
population are new, and the wording is clear that the first map counted fixed-route stops only. Run `python scripts\fetch_week3.py day18 day18b` first.

## 1 · Settings

In [ ]:
DAY      = 18
PROMPT   = "NULL"
SLUG     = "null-road-only-counties"          # same link as the first post, so it updates in place
NOTEBOOK = "day18_null_road_only_counties.ipynb"
TITLE    = "Public Transit Across the US"
UPDATED  = "Nov 20, 2026"
CREDITS  = ("Transit stops: BTS National Transit Map (NTAD). Amtrak stations and routes: BTS NTAD. Intercity bus, rail "
            "and ferry terminals: BTS Intermodal Passenger Connectivity Database. Dial-a-ride: FTA National Transit "
            "Database, Demand Response Geographic Area Coverage (Counties and Places; Passenger Eligibility and Fares). Counties and population: US Census "
            "Bureau (2023 counties, Population Estimates).")

COUNT_AIRPORTS = False     # True = a county with a commercial airport counts as connected

PAGE, INK, SOFT = "#efede8", "#1d1d1f", "#7a7a76"
ROAD, LINKED = "#2f3033", "#d9d6cf"        # asphalt: nothing reported · pale: has a fixed-route stop
DR = "#7f8c9a"                              # steel grey-blue: dial-a-ride only
RAIL = "#e0442b"                            # Amtrak, signal red
YELLOW = "#f2c230"                          # road-sign yellow outline: the train passes, no stop

DENSITY_BREAKS = [6, 20, 50, 150, 500]      # people per square mile of land
DENSITY_COLORS = ["#e3ecf1", "#bcd3e1", "#8db4cc", "#5e91b3", "#386d93", "#1d4a6b"]

## 2 · Load

In [ ]:
import json
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from mapkit import poster, publish
from mapkit.paths import PROCESSED, EQUAL_AREA

counties = gpd.read_file(PROCESSED / "counties_connectivity.geojson").to_crs(EQUAL_AREA)
routes = gpd.read_file(PROCESSED / "amtrak_routes.geojson").to_crs(EQUAL_AREA)
stations = gpd.read_file(PROCESSED / "amtrak_stations.geojson").to_crs(EQUAL_AREA)
dr = pd.read_csv(PROCESSED / "ntd_dr_counties.csv", dtype={"geoid": str})
pop = pd.read_csv(PROCESSED / "county_population.csv", dtype={"geoid": str})
dr["geoid"], pop["geoid"] = dr.geoid.str.zfill(5), pop.geoid.str.zfill(5)
_rep = json.loads((PROCESSED / "fetch_report.json").read_text())["day18b_dial_a_ride"]
POP_YEAR, NTD_YEAR = _rep["pop_year"], _rep["ntd_year"]

counties = counties.merge(dr, on="geoid", how="left").merge(pop, on="geoid", how="left")
counties["dr_agencies"] = counties.dr_agencies.fillna(0).astype(int)
counties["dr_names"] = counties.dr_names.fillna("")
counties["dr_open_agencies"] = counties.dr_open_agencies.fillna(0).astype(int)   # services with no rider eligibility rule
assert counties["pop"].notna().all(), "a county has no population: rerun fetch_week3.py day18b"
counties["density"] = counties["pop"] / counties.land_sqmi
print(f"{len(counties):,} counties · {len(stations):,} Amtrak stops · {len(routes)} Amtrak route pieces")
print(f"{(counties.dr_agencies > 0).sum():,} counties have dial-a-ride reported to the NTD · population year {POP_YEAR}")

## 3 · Analysis, every number on the map comes from here

In [ ]:
links = {
    "fixed-route transit": counties.transit_stops > 0,
    "Amtrak stop":         counties.amtrak_stations > 0,
    "intercity bus":       counties.ipcd_bus_i > 0,
    "rail terminal":       counties.ipcd_mode_rail > 0,
    "ferry":               counties.ipcd_mode_ferry > 0,
}
if COUNT_AIRPORTS:
    links["airport"] = counties.ipcd_mode_air > 0
counties["fixed"] = np.logical_or.reduce(list(links.values()))
counties["dr_only"] = ~counties.fixed & (counties.dr_agencies > 0)
counties["neither"] = ~counties.fixed & (counties.dr_agencies == 0)

# Counties an Amtrak line crosses without a fixed-route stop of any kind
rail_stops = stations[stations.StnType.str.upper().str.contains("RAIL", na=False)]
crossed = gpd.sjoin(counties[["geoid", "geometry"]], routes[["geometry"]], predicate="intersects").geoid.unique()
counties["train_passes"] = counties.geoid.isin(crossed)
counties["passed_by"] = ~counties.fixed & counties.train_passes

N, P = len(counties), counties["pop"].sum()
area = counties.area
S = dict(nofixed=int((~counties.fixed).sum()), dr=int(counties.dr_only.sum()), neither=int(counties.neither.sum()),
         fixed=int(counties.fixed.sum()), passed=int(counties.passed_by.sum()),
         pop_nofixed=int(counties.loc[~counties.fixed, "pop"].sum()), pop_neither=int(counties.loc[counties.neither, "pop"].sum()),
         area_nofixed=area[~counties.fixed].sum() / area.sum() * 100)
S["dr_open"] = int((counties.dr_only & (counties.dr_open_agencies > 0)).sum())
S["dr_restricted"] = S["dr"] - S["dr_open"]
assert S["dr"] + S["neither"] == S["nofixed"] and S["nofixed"] + S["fixed"] == N
print(f"{S['nofixed']:,} of {N:,} counties ({S['nofixed'] / N:.1%}) have no fixed-route stop, {S['area_nofixed']:.0f}% of the land")
print(f"   {S['dr']:,} of them have dial-a-ride reported to the NTD, {S['neither']:,} have neither")
print(f"   of the {S['dr']:,} dial-a-ride-only counties, {S['dr_open']:,} have a service open to anyone, "
      f"{S['dr_restricted']:,} only have services for eligible riders")
print(f"   {S['passed']} have an Amtrak line running through and no stop")
print(f"People: {S['pop_nofixed']:,} live in counties with no fixed-route stop ({S['pop_nofixed'] / P:.2%} of the Lower 48), "
      f"{S['pop_neither']:,} where neither is reported ({S['pop_neither'] / P:.2%})")
for k in ("fixed", "dr_only", "neither"):
    d = counties.loc[counties[k], "density"]
    print(f"   median density, {k}: {d.median():.1f} people per sq mi")
print("Density classes, all counties:", pd.cut(counties.density, [0] + DENSITY_BREAKS + [np.inf]).value_counts(sort=False).to_dict())
print("Most 'neither' counties by state:", counties[counties.neither].state_abb.value_counts().head(8).to_dict())

## 4 · Map

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.SLAB, poster.BARLOW, poster.BARLOW_BOLD     # Zilla Slab, like a road sign
fig, ax = poster.page(PAGE)
ax.set_position([0.02, 0.235, 0.96, 0.54])
counties[counties.fixed].plot(ax=ax, color=LINKED, edgecolor=PAGE, lw=0.15, zorder=1)
counties[counties.dr_only].plot(ax=ax, color=DR, edgecolor=PAGE, lw=0.15, zorder=2)
counties[counties.neither].plot(ax=ax, color=ROAD, edgecolor=PAGE, lw=0.15, zorder=2)
counties[counties.passed_by].boundary.plot(ax=ax, color=YELLOW, lw=0.9, zorder=3)
routes.plot(ax=ax, color=RAIL, lw=1.1, zorder=4)
ax.scatter(rail_stops.geometry.x, rail_stops.geometry.y, s=5, color=PAGE, edgecolor=RAIL, lw=0.6, zorder=5)
b = counties.total_bounds
ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_aspect("equal"); ax.set_axis_off()

fig.text(0.06, 0.955, "PUBLIC TRANSIT\nACROSS THE US", color=INK, fontproperties=F(TITLE_F, 54), va="top", linespacing=0.95)
fig.text(0.06, 0.842, f"{S['nofixed']:,} US counties have no stop for any scheduled bus, train or ferry, local or long-distance.",
         color=INK, fontproperties=F(TEXT_F, 15.5), va="top")
fig.text(0.06, 0.815, f"{S['dr']:,} of them have dial-a-ride. In {S['neither']:,}, home to {S['pop_neither'] / 1e6:.1f} million people, "
         f"neither is reported.", color=SOFT, fontproperties=F(TEXT_F, 15), va="top")

# legend, two columns
items = [(ROAD, "No fixed-route stop, no dial-a-ride reported", "fill"), (DR, "No fixed-route stop, dial-a-ride only", "fill"),
         (LINKED, "At least one fixed-route stop", "fill"), (YELLOW, "Amtrak passes through, no stop", "edge"),
         (RAIL, "Amtrak route", "line")]
for i, (col, lab, kind) in enumerate(items):
    lx, y = (0.06, 0.195 - i * 0.026) if i < 3 else (0.56, 0.195 - (i - 3) * 0.026)
    if kind == "line":
        fig.add_artist(plt.Line2D([lx, lx + 0.03], [y + 0.006] * 2, transform=fig.transFigure, color=col, lw=2))
    elif kind == "edge":
        fig.add_artist(plt.Rectangle((lx + 0.001, y + 0.001), 0.028, 0.011, transform=fig.transFigure, fc=PAGE, ec=col, lw=1.8))
    else:
        fig.add_artist(plt.Rectangle((lx, y), 0.03, 0.013, transform=fig.transFigure, color=col, lw=0))
    fig.text(lx + 0.042, y + 0.0065, lab, color=INK, fontproperties=F(TEXT_F, 12.5), va="center")

# credit, set by hand (Barlow sits taller than the helper expects)
for i, line in enumerate([f"Updated {UPDATED}: dial-a-ride and population added after your comments",
                          f"FTA National Transit Database, demand response coverage, {NTD_YEAR} · US Census 2023 counties, {POP_YEAR} population estimates",
                          "BTS National Transportation Atlas Database, accessed Nov 2026: National Transit Map, Amtrak, Intermodal Connectivity"][::-1]):
    fig.text(0.06, 0.020 + i * 0.017, line, color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.077, "Map by Gautami Kasat", color=INK, fontproperties=F(TEXT_B, 12), va="bottom")
plt.show()

## 4b · Interactive page: hover a county, switch between transit and people per square mile

In [ ]:
def _r(geom, nd=4):
    return json.loads(json.dumps(geom.__geo_interface__), parse_float=lambda x: round(float(x), nd))

web = counties.to_crs(4326)
status = np.where(counties.fixed, "some", np.where(counties.dr_only, "dr", "none"))
feats = []
for (_, r), st in zip(web.iterrows(), status):
    names = [n for n in r.dr_names.split("; ") if n]
    feats.append({"type": "Feature", "geometry": _r(r.geometry), "properties": {
        "n": r.county_name, "s": r.state_abb, "st": st, "pb": int(r.passed_by),
        "stops": int(r.transit_stops), "amtrak": int(r.amtrak_stations),
        "bus": int(r.ipcd_bus_i > 0), "ferry": int(r.ipcd_mode_ferry > 0),
        "pop": int(r["pop"]), "den": round(float(r.density), 1),
        "dr": int(r.dr_agencies), "op": int(r.dr_open_agencies), "drn": "; ".join(names[:3]) + (f" and {len(names) - 3} more" if len(names) > 3 else "")}})
web_routes = {"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": _r(g), "properties": {}}
                                                        for g in routes.to_crs(4326).geometry]}
web_stations = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": _r(r.geometry, 4), "properties": {"n": r.StationName, "t": str(r.StnType).title()}}
    for _, r in stations.to_crs(4326).iterrows()]}

def _fmt(x):
    return f"{x:,.0f}" if x >= 10 else f"{x:g}"
ramp_labels = [f"under {_fmt(DENSITY_BREAKS[0])}"] + [f"{_fmt(a)} to {_fmt(b)}" for a, b in zip(DENSITY_BREAKS, DENSITY_BREAKS[1:])] \
              + [f"{_fmt(DENSITY_BREAKS[-1])} or more"]
RAMP_HTML = "".join(f'<span class="sw" style="background:{c}"></span><span>{l}</span><span></span>'
                    for c, l in zip(DENSITY_COLORS, ramp_labels))
step_expr = ["step", ["get", "den"], DENSITY_COLORS[0]]
for b_, c in zip(DENSITY_BREAKS, DENSITY_COLORS[1:]):
    step_expr += [b_, c]

HTML = """<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__ · Gautami Kasat</title>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Zilla+Slab:wght@700&family=Barlow:wght@400;600&display=swap" rel="stylesheet">
<style>
  html,body{margin:0;height:100%;background:__PAGE__;font-family:Barlow,system-ui,sans-serif;color:__INK__}
  #map{position:absolute;inset:0}
  .panel{position:absolute;z-index:2;top:18px;left:18px;width:312px;max-width:calc(100% - 36px);max-height:calc(100% - 36px);overflow:auto;
         box-sizing:border-box;background:rgba(239,237,232,.95);border:1px solid #d4d1ca;border-radius:10px;padding:18px 20px 14px;
         box-shadow:0 6px 24px rgba(0,0,0,.08)}
  h1{font-family:'Zilla Slab',serif;font-weight:700;font-size:22px;line-height:1.05;margin:0 0 10px;text-transform:uppercase;letter-spacing:.3px}
  .big{font-family:'Zilla Slab',serif;font-weight:700;font-size:46px;line-height:1;margin:0}
  .bigl{font-size:14px;line-height:1.35;color:#4a4a47;margin:2px 0 12px}
  .tabs{display:flex;background:#e2dfd8;border-radius:7px;padding:3px;margin:0 0 10px}
  .tabs button{flex:1;border:0;background:none;font:600 13px Barlow,sans-serif;color:#5a5a56;padding:6px 4px;border-radius:5px;cursor:pointer}
  .tabs button.on{background:__PAGE__;color:__INK__;box-shadow:0 1px 3px rgba(0,0,0,.12)}
  .key{display:grid;grid-template-columns:22px 1fr auto;gap:6px 8px;align-items:center;font-size:13.5px;margin:4px 0 10px}
  .sw{width:20px;height:12px;border-radius:2px;box-sizing:border-box}
  .ln{width:20px;height:3px;background:__RAIL__;border-radius:2px}
  .n{font-weight:600;font-variant-numeric:tabular-nums}
  .keyhead{font-size:12.5px;color:__SOFT__;margin:0 0 2px}
  .stat{font-size:13.5px;line-height:1.4;margin:0 0 10px}
  .hint{font-size:12.5px;color:__SOFT__;margin:2px 0 8px}
  details{font-size:12.5px;color:#4a4a47;line-height:1.45;margin-bottom:10px}
  details p{margin:6px 0 0}
  summary{cursor:pointer;font-weight:600;color:__INK__}
  .credit{border-top:1px solid #d4d1ca;padding-top:8px;font-size:11.5px;color:__SOFT__;line-height:1.4}
  .credit b{display:block;color:__INK__;font-size:13px;margin-bottom:1px}
  .hide{display:none}
  .maplibregl-popup{z-index:5}
  .maplibregl-popup-content{font-family:Barlow,sans-serif;font-size:13.5px;padding:10px 12px;border-radius:8px;line-height:1.45;max-width:270px}
  .pop h3{margin:0 0 4px;font-size:15px}
  .pop .tag{display:inline-block;font-size:11.5px;font-weight:600;padding:1px 6px;border-radius:3px;margin-bottom:5px}
  .pop .row{display:flex;justify-content:space-between;gap:14px}
  .pop .row b{white-space:nowrap}
  .pop .no{color:#9a9893}
  .pop .drn{font-size:12px;color:#5a5a56;line-height:1.35;margin:1px 0 2px}
  .pop .pass{font-size:12px;font-weight:600;margin-top:4px}
  .pop hr{border:0;border-top:1px solid #e2dfd8;margin:5px 0}
  .nw{white-space:nowrap}
  @media (max-width:600px){.panel{width:auto;right:10px;left:10px;top:auto;bottom:10px;padding:12px 14px;max-height:50%}
    h1{font-size:19px;margin-bottom:6px}.big{font-size:34px}.bigl{margin-bottom:8px}.hint{display:none}}
</style></head><body>
<div id="map"></div>
<div class="panel">
  <h1>__TITLE__</h1>
  <div class="big">__N_FIXED__</div>
  <div class="bigl">of __N_ALL__ counties in the Lower 48 have a fixed-route bus, train or ferry stop</div>
  <div class="tabs"><button id="t-access" class="on">Transit</button><button id="t-density">People per sq mi</button></div>
  <div id="k-access" class="key">
    <span class="sw" style="background:__ROAD__"></span><span>No fixed-route, no dial-a-ride</span><span class="n">__N_NEITHER__</span>
    <span class="sw" style="background:__DR__"></span><span>Dial-a-ride only</span><span class="n">__N_DR__</span>
    <span class="sw" style="background:__LINKED__;border:1px solid #c9c5bc"></span><span>Has fixed-route transit</span><span class="n">__N_FIXED__</span>
    <span class="sw" style="border:2px solid __YELLOW__"></span><span>Amtrak passes, no stop</span><span class="n">__N_PASSED__</span>
    <span class="ln"></span><span>Amtrak route</span><span></span>
  </div>
  <div id="k-density" class="hide">
    <div class="keyhead">People per square mile of land, __POP_YEAR__</div>
    <div class="key">__RAMP__
      <span class="sw" style="border:1.5px solid __INK__"></span><span>Outlined: no fixed-route stop</span><span></span></div>
  </div>
  <p class="stat"><b>__POP_NOFIXED__ million</b> people live in counties with no fixed-route stop. <b>__POP_NEITHER__ million</b>
    of them live where no dial-a-ride is reported either.</p>
  <p class="hint">Hover a county for details</p>
  <details><summary>How this is counted</summary>
    <p><b>Fixed-route:</b> at least one stop on a scheduled local or regional bus, light rail, subway or commuter rail route
    (National Transit Map), an Amtrak station or Amtrak bus stop, or an intercity bus, rail or ferry terminal such as
    Greyhound or Trailways (Intermodal Passenger Connectivity Database).</p>
    <p><b>Dial-a-ride:</b> at least one agency reported demand-response service in the county to the National Transit
    Database for __NTD_YEAR__. It may cover only part of the county and usually needs booking ahead. "Open to anyone"
    means the agency reports no rider eligibility rule; the rest are for riders who qualify, such as older adults or
    riders with disabilities.</p>
    <p><b>Not reported isn't the same as nothing.</b> Volunteer drivers, senior center vans and Medicaid rides mostly don't
    report to the NTD, and the National Transit Map only has agencies that share their schedules. School buses and
    airports are not counted.</p></details>
  <div class="credit"><b>Map by Gautami Kasat</b>__CREDITS__</div>
</div>
<script>
const COUNTIES = __COUNTIES__, ROUTES = __ROUTES__, STATIONS = __STATIONS__;
const ACCESS = ["match",["get","st"],"none","__ROAD__","dr","__DR__","__LINKED__"], DENSITY = __STEP__;
const map = new maplibregl.Map({container:"map", attributionControl:false,
  style:{version:8, sources:{}, layers:[{id:"bg", type:"background", paint:{"background-color":"__PAGE__"}}]},
  bounds:[[-125,24.3],[-66.9,49.4]], fitBoundsOptions:{padding: window.innerWidth > 700 ? {top:30,bottom:30,left:400,right:30}
    : {top:16,bottom:Math.round(window.innerHeight * 0.52),left:8,right:8}}});
map.addControl(new maplibregl.NavigationControl({showCompass:false}), window.innerWidth > 700 ? "bottom-right" : "top-right");
map.on("load", () => {
  map.addSource("c", {type:"geojson", data:COUNTIES, generateId:true});
  map.addLayer({id:"fill", type:"fill", source:"c", paint:{"fill-color":ACCESS,
    "fill-opacity":["case",["boolean",["feature-state","hover"],false],0.75,1]}});
  map.addLayer({id:"edge", type:"line", source:"c", paint:{"line-color":"__PAGE__","line-width":0.35}});
  map.addLayer({id:"passed", type:"line", source:"c", filter:["==",["get","pb"],1],
    paint:{"line-color":"__YELLOW__","line-width":["interpolate",["linear"],["zoom"],3,1,8,2.4]}});
  map.addLayer({id:"nofixed", type:"line", source:"c", filter:["!=",["get","st"],"some"], layout:{visibility:"none"},
    paint:{"line-color":"__INK__","line-width":["interpolate",["linear"],["zoom"],3,0.5,8,1.4]}});
  map.addSource("r", {type:"geojson", data:ROUTES});
  map.addLayer({id:"rail", type:"line", source:"r", paint:{"line-color":"__RAIL__","line-width":["interpolate",["linear"],["zoom"],3,1.2,8,2.6]}});
  map.addSource("s", {type:"geojson", data:STATIONS});
  map.addLayer({id:"stn", type:"circle", source:"s", minzoom:5, paint:{"circle-radius":3,"circle-color":"__PAGE__","circle-stroke-color":"__RAIL__","circle-stroke-width":1.2}});
  map.addLayer({id:"hl", type:"line", source:"c", paint:{"line-color":"__INK__","line-width":["case",["boolean",["feature-state","hover"],false],1.8,0]}});

  const show = mode => {
    const d = mode === "density";
    map.setPaintProperty("fill", "fill-color", d ? DENSITY : ACCESS);
    map.setLayoutProperty("passed", "visibility", d ? "none" : "visible");
    map.setLayoutProperty("nofixed", "visibility", d ? "visible" : "none");
    document.getElementById("k-access").classList.toggle("hide", d);
    document.getElementById("k-density").classList.toggle("hide", !d);
    document.getElementById("t-access").classList.toggle("on", !d);
    document.getElementById("t-density").classList.toggle("on", d);
  };
  document.getElementById("t-access").onclick = () => show("access");
  document.getElementById("t-density").onclick = () => show("density");

  const pop = new maplibregl.Popup({closeButton:false, closeOnClick:false, offset:10});
  let hov = null;
  const num = x => x.toLocaleString("en-US");
  const row = (label, val, on) => `<div class="row"><span class="${on?'':'no'}">${label}</span><b class="${on?'':'no'}">${val}</b></div>`;
  const card = e => {
    const f = e.features[0], p = f.properties;
    if (hov !== null) map.setFeatureState({source:"c", id:hov}, {hover:false});
    hov = f.id; map.setFeatureState({source:"c", id:hov}, {hover:true});
    map.getCanvas().style.cursor = "pointer";
    const tag = p.st === "some" ? ["__LINKED__","__INK__","Has fixed-route transit"] :
                p.st === "dr" ? ["__DR__","#fff","No fixed-route stop · dial-a-ride only"] :
                ["__ROAD__","#fff","No fixed-route stop · no dial-a-ride reported"];
    pop.setLngLat(e.lngLat).setHTML(`<div class="pop"><h3>${p.n}, ${p.s}</h3>
      <span class="tag" style="background:${tag[0]};color:${tag[1]}">${tag[2]}</span>
      ${row("People", num(p.pop), true)}${row("Per sq mi", p.den < 10 ? p.den.toFixed(1) : num(Math.round(p.den)), true)}<hr>
      ${row("Fixed-route stops", num(p.stops), p.stops)}${row("Amtrak stops", p.amtrak, p.amtrak)}
      ${row("Intercity bus terminal", p.bus ? "yes" : "none", p.bus)}${row("Ferry terminal", p.ferry ? "yes" : "none", p.ferry)}
      ${row("Dial-a-ride agencies (NTD)", p.dr || "none", p.dr)}
      ${p.dr ? `<div class="drn">${p.drn}</div>` + row("Who can ride", p.op ? (p.dr > 1 ? `anyone (${p.op} of ${p.dr})` : "anyone") : "eligible riders", true) : ""}
      ${p.pb ? `<div class="pass">An Amtrak line runs through, with no stop</div>` : ""}</div>`).addTo(map);
  };
  map.on("mousemove", "fill", card);
  map.on("click", "fill", card);          // phones: tap a county
  map.on("mouseleave", "fill", () => { if (hov !== null) map.setFeatureState({source:"c", id:hov}, {hover:false}); hov = null;
                                      pop.remove(); map.getCanvas().style.cursor = ""; });
  const sp = new maplibregl.Popup({closeButton:false, offset:8});
  map.on("mouseenter", "stn", e => { const p = e.features[0].properties;
    sp.setLngLat(e.features[0].geometry.coordinates).setHTML(`<b>${p.n}</b><br>Amtrak ${p.t.toLowerCase()} stop`).addTo(map); });
  map.on("mouseleave", "stn", () => sp.remove());
});
</script></body></html>"""

fill = {"__TITLE__": TITLE, "__PAGE__": PAGE, "__INK__": INK, "__SOFT__": SOFT, "__ROAD__": ROAD, "__DR__": DR,
        "__LINKED__": LINKED, "__RAIL__": RAIL, "__YELLOW__": YELLOW, "__UPDATED__": UPDATED, "__POP_YEAR__": str(POP_YEAR), "__NTD_YEAR__": str(NTD_YEAR),
        "__N_NOFIXED__": f"{S['nofixed']:,}", "__N_ALL__": f"{N:,}", "__N_NEITHER__": f"{S['neither']:,}",
        "__N_DR__": f"{S['dr']:,}", "__N_DR_OPEN__": f"{S['dr_open']:,}", "__N_FIXED__": f"{S['fixed']:,}", "__N_PASSED__": f"{S['passed']:,}",
        "__POP_NOFIXED__": f"{S['pop_nofixed'] / 1e6:.1f}", "__POP_NEITHER__": f"{S['pop_neither'] / 1e6:.1f}",
        "__RAMP__": RAMP_HTML, "__STEP__": json.dumps(step_expr),
        "__CREDITS__": f"Data: BTS NTAD (2026) · FTA National Transit Database ({NTD_YEAR}) · US Census ({POP_YEAR})",
        "__COUNTIES__": json.dumps({"type": "FeatureCollection", "features": feats}, separators=(",", ":")),
        "__ROUTES__": json.dumps(web_routes, separators=(",", ":")),
        "__STATIONS__": json.dumps(web_stations, separators=(",", ":"))}
page = HTML
for k, v in fill.items():
    page = page.replace(k, v)
import re
assert not re.search(r"__[A-Z_]+__", page), "a placeholder was left in the page"
print(f"page size {len(page) / 1e6:.1f} MB")

## 5 · Export: the interactive page (live map + gallery), and the still poster for LinkedIn

In [ ]:
from mapkit.paths import OUTPUT
publish.save(page, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE,
             subtitle=(f"{S['nofixed']:,} US counties with no fixed-route bus, train or ferry stop; "
                       f"{S['dr']:,} have dial-a-ride, {S['neither']:,} report neither"),
             series="city", tool="Python + MapLibre", notebook=NOTEBOOK, credits=CREDITS)
fig.savefig(OUTPUT / f"day{DAY:02d}_{SLUG}_poster.png", dpi=100, facecolor=PAGE)
print("poster     output/" + f"day{DAY:02d}_{SLUG}_poster.png")